In [23]:
import pandas as pd
from collections import defaultdict
import pickle
import os

In [ ]:
from dotenv import find_dotenv,load_dotenv

def set_env():
    env_file = find_dotenv()
    if env_file:
        load_dotenv(env_file,override=True)
    else:
        raise Exception("No valid .env file found. See env_template at project root for required env vars")

set_env()

In [25]:
data_base_path=os.getcwd().split("clincodex")[0]+"/data/"
data_base_path

'/Users/Sankalp.C/Desktop/Neo/clincodex//data/'

In [114]:
with open(data_base_path+'parent_child_dict.pkl', 'rb') as f:
    parent_child_dict = pickle.load(f)

In [27]:
import pickle
with open(os.getcwd().split("clincodex")[0]+"/data/disease.pkl", 'rb') as f:
    disease_list = pickle.load(f)

In [67]:
with open(data_base_path+"icd10cm-order-April-2024.txt", 'r') as file:
        lines = file.readlines()
    
base_data_dict={}
parent_child_dict=defaultdict(list)

for line in lines:
    code=line[6:14].strip()
    descripts=line[77:].strip()

    base_data_dict[code]=descripts

In [28]:
context_str=disease_list

In [29]:
print(context_str)

['Cholera || A00 ', 'Typhoid and paratyphoid fevers || A01 ', 'Other salmonella infections || A02 ', 'Shigellosis || A03 ', 'Other bacterial intestinal infections || A04 ', 'Other bacterial foodborne intoxications, not elsewhere classified || A05 ', 'Amebiasis || A06 ', 'Other protozoal intestinal diseases || A07 ', 'Viral and other specified intestinal infections || A08 ', 'Infectious gastroenteritis and colitis, unspecified || A09 ', 'Respiratory tuberculosis || A15 ', 'Tuberculosis of nervous system || A17 ', 'Tuberculosis of other organs || A18 ', 'Miliary tuberculosis || A19 ', 'Plague || A20 ', 'Tularemia || A21 ', 'Anthrax || A22 ', 'Brucellosis || A23 ', 'Glanders and melioidosis || A24 ', 'Rat-bite fevers || A25 ', 'Erysipeloid || A26 ', 'Leptospirosis || A27 ', 'Other zoonotic bacterial diseases, not elsewhere classified || A28 ', "Leprosy [Hansen's disease] || A30 ", 'Infection due to other mycobacteria || A31 ', 'Listeriosis || A32 ', 'Tetanus neonatorum || A33 ', 'Obstetri

In [30]:
import re 

user_input='''The patient is a 66-year-old female who presents to the clinic today for a five-month recheck on her
                adult onset diabetes mellitus along with nerve damage,
                as well as
                hypertension.
                While here she had a couple of other issues as well. 
                She stated that she has been having some 
                right shoulder pain.
                She denies any injury but certain range of motion does cause it to hurt. 
                No weakness, numbness or tingling.
                As far as her 
                diabetes
                she states that she only checks her blood sugars in the morning and those have all been ranging less than 100. 
                She has not been checking any two hours after meals. 
                Since childhood she has also been suffering from mild ashtama that is persistent
                Her
                blood pressures
                when she does check them have been running normal as well but she does not have any record of these present with her. 
                No other issues or concerns.
                One other associated problem was edema that was of macular type and present in both left and right eyes.
                Upon review of her chart it did show that she had a 
                benign breast biopsy for left breast done back on 06/11/04 and was told to have a repeat 
                mammogram in six months but she has never had that done so she is needing to have this done as well.
                The patient has been also suffering from mild ckd disease.
                She had successful treatment for macular edema two years ago.'''.replace("\n"," ")#.split(".")

#txt_list=[re.sub(r'\s+', ' ', x) for x in txt_list if x]

In [31]:
# patient has adult onset diabetes with extensive nerve damage. had bipolar macular edema successfully treated last year

In [116]:
user_input = re.sub(r'\s+', ' ', user_input)

In [117]:
user_input

'The patient is a 66-year-old female who presents to the clinic today for a five-month recheck on her adult onset diabetes mellitus along with nerve damage, as well as hypertension. While here she had a couple of other issues as well. She stated that she has been having some right shoulder pain. She denies any injury but certain range of motion does cause it to hurt. No weakness, numbness or tingling. As far as her diabetes she states that she only checks her blood sugars in the morning and those have all been ranging less than 100. She has not been checking any two hours after meals. Since childhood she has also been suffering from mild ashtama that is persistent Her blood pressures when she does check them have been running normal as well but she does not have any record of these present with her. No other issues or concerns. One other associated problem was edema that was of macular type and present in both left and right eyes. Upon review of her chart it did show that she had a ben

In [33]:
import boto3
import os
import json

import re
import ast
import copy

import yaml

import logging

boto3_logger = logging.getLogger('boto3')
botocore_logger = logging.getLogger('botocore')

boto3_logger.setLevel(logging.ERROR)
botocore_logger.setLevel(logging.ERROR)

class claudeSonnet:
    _client = None

    def get_client(self):
        if claudeSonnet._client is not None:
            return claudeSonnet._client
        else:
            ANTHROPIC_RESOURCE_ENDPOINT =os.environ["ANTHROPIC_RESOURCE_ENDPOINT"]
            ANTRHOPIC_ACCESS_KEY_ID = os.environ["ANTRHOPIC_ACCESS_KEY_ID"]
            ANTRHOPIC_SECRET_ACCESS_KEY = os.environ["ANTRHOPIC_SECRET_ACCESS_KEY"]
            ANTHROPIC_REGION=os.environ["ANTHROPIC_REGION"]

            claudeSonnet._client = boto3.client('bedrock-runtime',
                                                endpoint_url=ANTHROPIC_RESOURCE_ENDPOINT,
                                                aws_access_key_id=ANTRHOPIC_ACCESS_KEY_ID,
                                                aws_secret_access_key=ANTRHOPIC_SECRET_ACCESS_KEY,
                                                region_name=ANTHROPIC_REGION
                                            )
    
        return claudeSonnet._client
    
    @staticmethod
    def load_yaml_from_string(yaml_string):
        try:
            # Load the YAML string into a Python dictionary
            data = yaml.safe_load(yaml_string)
            return data
        except yaml.YAMLError as e:
            print(f"Error loading YAML: {e}")
            return None

    def generate_message(self,bedrock_runtime, model_id, system_prompt, messages, max_tokens, temperature=1):
        body=json.dumps(
            {
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": max_tokens,
                "system": system_prompt,
                "temperature": temperature,
                "messages": messages
            }  
        )  
        response = bedrock_runtime.invoke_model(body=body, modelId=model_id)
        response_body = json.loads(response.get('body').read())
    
        return response_body
    
    def claude_wrapper(self,system_prompt, user_input, temperature=0.5):

        model_id = os.environ["ANTHROPIC_MODEL_ID"]
        max_tokens = 4000

        user_message =  {"role": "user", "content": user_input}
        messages = [user_message]

        response = self.generate_message(self.get_client(), model_id, system_prompt, messages, max_tokens, temperature=temperature) 
        print(response['usage'])
        out = response['content'][0]['text']


        return out
    
client = claudeSonnet()

In [91]:
with open(os.getcwd().split("clincodex")[0]+"/system_prompts_v2.yaml", 'r') as f:
            system_prompts = yaml.safe_load(f)

system_prompt=system_prompts["DISEASE_ENTITY_EXTRACTION"]

In [83]:
print(system_prompt)

You are an expert in disease entity extraction from a given list of sentences. 
Your task is to extract disease entities in python dict format as given in output format below.

<<< Requirements >>>

1. Extract all disease entities mentioned in the text that are also present in the list of diseases that is provided.
2. Ensure that all the relevant disease entities are extracted to prevent potential losses for the hospital.

<<< Input Format for disease list>>>

[ "<Disease || Code>", "<Disease || Code>", .. ]

<<< Output Format >>>

{
  "<Disease || Code>": "<Evidence of disease exactly as present in text>"
}

 << Process >>

1. Analyze each sentence in the list individually and extract disease entities. 
2. Try to map the disease semantically to a entry in the list of diseases that is provided.
3. Discrad all extracted diseases that can't be confidently mapped to any entity in the list provided.

<< Most Important Conditions >>

1. Key of the python dict should necessarily be a part of

In [63]:
import json

ner_agent_res=json.loads(client.claude_wrapper(system_prompt,user_input=f'''
                <<Text List >>
                {user_input}
        
                << Disease List >>
                {context_str}
                
                '''))

{'input_tokens': 32386, 'output_tokens': 84}


In [73]:
coder_agent_pack={}

for key,value in ner_agent_res.items():
    descript,code=key.split("||")
    coder_agent_pack[code.strip()] = [descript,value]

In [74]:
coder_agent_pack

{'E11': ['Type 2 diabetes mellitus ', 'adult onset diabetes mellitus'],
 'N18': ['Chronic kidney disease (CKD) ', 'ckd disease'],
 'I10': ['Essential (primary) hypertension ', 'hypertension'],
 'J45': ['Asthma ', 'ashtama']}

In [ ]:
from graphviz import Digraph

def medgrapher(t2dm_data_list):
    graph = Digraph(format='png', node_attr={'shape': 'box', 'style': 'rounded'})

    for code, description in t2dm_data_list.items():
        if all(child[:len(code)] != code for child in t2dm_data_list if child != code):
            graph.node(code, f"{code}: {description} (billable)", style="filled", fillcolor="lightblue")
        else:
            graph.node(code, f"{code}: {description}")
        if len(code) > 3:
            graph.edge(code[:len(code)-1], code)
            
    return graph

# graph.render("diabetes_hierarchy", view=True)

In [76]:
for key,value in coder_agent_pack.items():
    t2dm_data_list={k:v for k,v in base_data_dict.items() if k.startswith(key)}

    value.append(medgrapher(t2dm_data_list))

In [77]:
coder_agent_pack

{'E11': ['Type 2 diabetes mellitus ',
  'adult onset diabetes mellitus',
 'N18': ['Chronic kidney disease (CKD) ',
  'ckd disease',
 'I10': ['Essential (primary) hypertension ',
  'hypertension',
 'J45': ['Asthma ', 'ashtama', <graphviz.graphs.Digraph at 0x1284102e0>]}

In [80]:
print(coder_agent_pack['N18'][2])

digraph {
	node [shape=box style=rounded]
	N18 [label="N18: Chronic kidney disease (CKD)"]
	N181 [label="N181: Chronic kidney disease, stage 1 (billable)" fillcolor=lightblue style=filled]
	N18 -> N181
	N182 [label="N182: Chronic kidney disease, stage 2 (mild) (billable)" fillcolor=lightblue style=filled]
	N18 -> N182
	N183 [label="N183: Chronic kidney disease, stage 3 (moderate)"]
	N18 -> N183
	N1830 [label="N1830: Chronic kidney disease, stage 3 unspecified (billable)" fillcolor=lightblue style=filled]
	N183 -> N1830
	N1831 [label="N1831: Chronic kidney disease, stage 3a (billable)" fillcolor=lightblue style=filled]
	N183 -> N1831
	N1832 [label="N1832: Chronic kidney disease, stage 3b (billable)" fillcolor=lightblue style=filled]
	N183 -> N1832
	N184 [label="N184: Chronic kidney disease, stage 4 (severe) (billable)" fillcolor=lightblue style=filled]
	N18 -> N184
	N185 [label="N185: Chronic kidney disease, stage 5 (billable)" fillcolor=lightblue style=filled]
	N18 -> N185
	N186 [label

In [ ]:
'''E11: Type 2 diabetes mellitus
├── E110: Type 2 diabetes mellitus with hyperosmolarity
│   ├── E1100: Type 2 diabetes mellitus with hyperosmolarity without nonketotic hyperglycemic-hyperosmolar coma (NKHHC)
│   └── E1101: Type 2 diabetes mellitus with hyperosmolarity with coma
├── E111: Type 2 diabetes mellitus with ketoacidosis
│   ├── E1110: Type 2 diabetes mellitus with ketoacidosis without coma
│   └── E1111: Type 2 diabetes mellitus with ketoacidosis with coma
├── E112: Type 2 diabetes mellitus with kidney complications
│   ├── E1121: Type 2 diabetes mellitus with diabetic nephropathy
│   ├── E1122: Type 2 diabetes mellitus with diabetic chronic kidney disease
│   └── E1129: Type 2 diabetes mellitus with other diabetic kidney complication
├── E113: Type 2 diabetes mellitus with ophthalmic complications
│   ├── E1131: Type 2 diabetes mellitus with unspecified diabetic retinopathy
│   │   ├── E11311: Type 2 diabetes mellitus with unspecified diabetic retinopathy with macular edema
│   │   └── E11319: Type 2 diabetes mellitus with unspecified diabetic retinopathy without macular edema
│   ├── E1132: Type 2 diabetes mellitus with mild nonproliferative diabetic retinopathy
│   │   ├── E11321: Type 2 diabetes mellitus with mild nonproliferative diabetic retinopathy with macular edema
│   │   │   ├── E113211: Type 2 diabetes mellitus with mild nonproliferative diabetic retinopathy with macular edema, right eye
│   │   │   ├── E113212: Type 2 diabetes mellitus with mild nonproliferative diabetic retinopathy with macular edema, left eye
│   │   │   ├── E113213: Type 2 diabetes mellitus with mild nonproliferative diabetic retinopathy with macular edema, bilateral
│   │   │   └── E113219: Type 2 diabetes mellitus with mild nonproliferative diabetic retinopathy with macular edema, unspecified eye
│   │   └── E11329: Type 2 diabetes mellitus with mild nonproliferative diabetic retinopathy without macular edema
│   │       ├── E113291: Type 2 diabetes mellitus with mild nonproliferative diabetic retinopathy without macular edema, right eye
│   │       ├── E113292: Type 2 diabetes mellitus with mild nonproliferative diabetic retinopathy without macular edema, left eye
│   │       ├── E113293: Type 2 diabetes mellitus with mild nonproliferative diabetic retinopathy without macular edema, bilateral
│   │       └── E113299: Type 2 diabetes mellitus with mild nonproliferative diabetic retinopathy without macular edema, unspecified eye
│   ├── E1133: Type 2 diabetes mellitus with moderate nonproliferative diabetic retinopathy
│   │   ├── E11331: Type 2 diabetes mellitus with moderate nonproliferative diabetic retinopathy with macular edema
│   │   │   ├── E113311: Type 2 diabetes mellitus with moderate nonproliferative diabetic retinopathy with macular edema, right eye
│   │   │   ├── E113312: Type 2 diabetes mellitus with moderate nonproliferative diabetic retinopathy with macular edema, left eye
│   │   │   ├── E113313: Type 2 diabetes mellitus with moderate nonproliferative diabetic retinopathy with macular edema, bilateral
│   │   │   └── E113319: Type 2 diabetes mellitus with moderate nonproliferative diabetic retinopathy with macular edema, unspecified eye
│   │   └── E11339: Type 2 diabetes mellitus with moderate nonproliferative diabetic retinopathy without macular edema
│   │       ├── E113391: Type 2 diabetes mellitus with moderate nonproliferative diabetic retinopathy without macular edema, right eye
│   │       ├── E113392: Type 2 diabetes mellitus with moderate nonproliferative diabetic retinopathy without macular edema, left eye
│   │       ├── E113393: Type 2 diabetes mellitus with moderate nonproliferative diabetic retinopathy without macular edema, bilateral
│   │       └── E113399: Type 2 diabetes mellitus with moderate nonproliferative diabetic retinopathy without macular edema, unspecified eye
│   ├── E1134: Type 2 diabetes mellitus with severe nonproliferative diabetic retinopathy
│   │   ├── E11341: Type 2 diabetes mellitus with severe nonproliferative diabetic retinopathy with macular edema
│   │   │   ├── E113411: Type 2 diabetes mellitus with severe nonproliferative diabetic retinopathy with macular edema, right eye
│   │   │   ├── E113412: Type 2 diabetes mellitus with severe nonproliferative diabetic retinopathy with macular edema, left eye
│   │   │   ├── E113413: Type 2 diabetes mellitus with severe nonproliferative diabetic retinopathy with macular edema, bilateral
│   │   │   └── E113419: Type 2 diabetes mellitus with severe nonproliferative diabetic retinopathy with macular edema, unspecified eye
│   │   └── E11349: Type 2 diabetes mellitus with severe nonproliferative diabetic retinopathy without macular edema
│   │       ├── E113491: Type 2 diabetes mellitus with severe nonproliferative diabetic retinopathy without macular edema, right eye
│   │       ├── E113492: Type 2 diabetes mellitus with severe nonproliferative diabetic retinopathy without macular edema, left eye
│   │       ├── E113493: Type 2 diabetes mellitus with severe nonproliferative diabetic retinopathy without macular edema, bilateral
│   │       └── E113499: Type 2 diabetes mellitus with severe nonproliferative diabetic retinopathy without macular edema, unspecified eye
│   ├── E1135: Type 2 diabetes mellitus with proliferative diabetic retinopathy
│   │   ├── E11351: Type 2 diabetes mellitus with proliferative diabetic retinopathy with macular edema
│   │   │   ├── E113511: Type 2 diabetes mellitus with proliferative diabetic retinopathy with macular edema, right eye
│   │   │   ├── E113512: Type 2 diabetes mellitus with proliferative diabetic retinopathy with macular edema, left eye
│   │   │   ├── E113513: Type 2 diabetes mellitus with proliferative diabetic retinopathy with macular edema, bilateral
│   │   │   └── E113519: Type 2 diabetes mellitus with proliferative diabetic retinopathy with macular edema, unspecified eye
│   │   ├── E11352: Type 2 diabetes mellitus with proliferative diabetic retinopathy with traction retinal detachment involving the macula
│   │   │   ├── E113521: Type 2 diabetes mellitus with proliferative diabetic retinopathy with traction retinal detachment involving the macula, right eye
│   │   │   ├── E113522: Type 2 diabetes mellitus with proliferative diabetic retinopathy with traction retinal detachment involving the macula, left eye
│   │   │   ├── E113523: Type 2 diabetes mellitus with proliferative diabetic retinopathy with traction retinal detachment involving the macula, bilateral
│   │   │   └── E113529: Type 2 diabetes mellitus with proliferative diabetic retinopathy with traction retinal detachment involving the macula, unspecified eye
│   │   ├── E11353: Type 2 diabetes mellitus with proliferative diabetic retinopathy with traction retinal detachment not involving the macula
│   │   │   ├── E113531: Type 2 diabetes mellitus with proliferative diabetic retinopathy with traction retinal detachment not involving the macula, right eye
│   │   │   ├── E113532: Type 2 diabetes mellitus with proliferative diabetic retinopathy with traction retinal detachment not involving the macula, left eye
│   │   │   ├── E113533: Type 2 diabetes mellitus with proliferative diabetic retinopathy with traction retinal detachment not involving the macula, bilateral
│   │   │   └── E113539: Type 2 diabetes mellitus with proliferative diabetic retinopathy with traction retinal detachment not involving the macula, unspecified eye
│   │   ├── E11354: Type 2 diabetes mellitus with proliferative diabetic retinopathy with combined traction retinal detachment and rhegmatogenous retinal detachment
│   │   │   ├── E113541: Type 2 diabetes mellitus with proliferative diabetic retinopathy with combined traction retinal detachment and rhegmatogenous retinal detachment, right eye
│   │   │   ├── E113542: Type 2 diabetes mellitus with proliferative diabetic retinopathy with combined traction retinal detachment and rhegmatogenous retinal detachment, left eye
│   │   │   ├── E113543: Type 2 diabetes mellitus with proliferative diabetic retinopathy with combined traction retinal detachment and rhegmatogenous retinal detachment, bilateral
│   │   │   └── E113549: Type 2 diabetes mellitus with proliferative diabetic retinopathy with combined traction retinal detachment and rhegmatogenous retinal detachment, unspecified eye
│   │   ├── E11355: Type 2 diabetes mellitus with stable proliferative diabetic retinopathy
│   │   │   ├── E113551: Type 2 diabetes mellitus with stable proliferative diabetic retinopathy, right eye
│   │   │   ├── E113552: Type 2 diabetes mellitus with stable proliferative diabetic retinopathy, left eye
│   │   │   ├── E113553: Type 2 diabetes mellitus with stable proliferative diabetic retinopathy, bilateral
│   │   │   └── E113559: Type 2 diabetes mellitus with stable proliferative diabetic retinopathy, unspecified eye
│   │   ├── E11359: Type 2 diabetes mellitus with proliferative diabetic retinopathy without macular edema
│   │   │   ├── E113591: Type 2 diabetes mellitus with proliferative diabetic retinopathy without macular edema, right eye
│   │   │   ├── E113592: Type 2 diabetes mellitus with proliferative diabetic retinopathy without macular edema, left eye
│   │   │   ├── E113593: Type 2 diabetes mellitus with proliferative diabetic retinopathy without macular edema, bilateral
│   │   │   └── E113599: Type 2 diabetes mellitus with proliferative diabetic retinopathy without macular edema, unspecified eye
│   ├── E1136: Type 2 diabetes mellitus with diabetic cataract
│   ├── E1137: Type 2 diabetes mellitus with diabetic macular edema, resolved following treatment
│   │   ├── E1137X1: Type 2 diabetes mellitus with diabetic macular edema, resolved following treatment, right eye
│   │   ├── E1137X2: Type 2 diabetes mellitus with diabetic macular edema, resolved following treatment, left eye
│   │   ├── E1137X3: Type 2 diabetes mellitus with diabetic macular edema, resolved following treatment, bilateral
│   │   └── E1137X9: Type 2 diabetes mellitus with diabetic macular edema, resolved following treatment, unspecified eye
│   └── E1139: Type 2 diabetes mellitus with other diabetic ophthalmic complication
├── E114: Type 2 diabetes mellitus with neurological complications
│   ├── E1140: Type 2 diabetes mellitus with diabetic neuropathy, unspecified
│   ├── E1141: Type 2 diabetes mellitus with diabetic mononeuropathy
│   ├── E1142: Type 2 diabetes mellitus with diabetic polyneuropathy
│   ├── E1143: Type 2 diabetes mellitus with diabetic autonomic (poly)neuropathy
│   ├── E1144: Type 2 diabetes mellitus with diabetic amyotrophy
│   └── E1149: Type 2 diabetes mellitus with other diabetic neurological complication
├── E115: Type 2 diabetes mellitus with circulatory complications
│   ├── E1151: Type 2 diabetes mellitus with diabetic peripheral angiopathy without gangrene
│   ├── E1152: Type 2 diabetes mellitus with diabetic peripheral angiopathy with gangrene
│   └── E1159: Type 2 diabetes mellitus with other circulatory complications
├── E116: Type 2 diabetes mellitus with other specified complications
│   ├── E1161: Type 2 diabetes mellitus with diabetic arthropathy
│   │   ├── E11610: Type 2 diabetes mellitus with diabetic neuropathic arthropathy
│   │   └── E11618: Type 2 diabetes mellitus with other diabetic arthropathy
│   ├── E1162: Type 2 diabetes mellitus with skin complications
│   │   ├── E11620: Type 2 diabetes mellitus with diabetic dermatitis
│   │   ├── E11621: Type 2 diabetes mellitus with foot ulcer
│   │   ├── E11622: Type 2 diabetes mellitus with other skin ulcer
│   │   └── E11628: Type 2 diabetes mellitus with other skin complications
│   ├── E1163: Type 2 diabetes mellitus with oral complications
│   │   ├── E11630: Type 2 diabetes mellitus with periodontal disease
│   │   └── E11638: Type 2 diabetes mellitus with other oral complications
│   ├── E1164: Type 2 diabetes mellitus with hypoglycemia
│   │   ├── E11641: Type 2 diabetes mellitus with hypoglycemia with coma
│   │   └── E11649: Type 2 diabetes mellitus with hypoglycemia without coma
│   ├── E1165: Type 2 diabetes mellitus with hyperglycemia
│   └── E1169: Type 2 diabetes mellitus with other specified complication
├── E118: Type 2 diabetes mellitus with unspecified complications
└── E119: Type 2 diabetes mellitus without complications
'''

'E11: Type 2 diabetes mellitus\n├── E110: Type 2 diabetes mellitus with hyperosmolarity\n│   ├── E1100: Type 2 diabetes mellitus with hyperosmolarity without nonketotic hyperglycemic-hyperosmolar coma (NKHHC)\n│   └── E1101: Type 2 diabetes mellitus with hyperosmolarity with coma\n├── E111: Type 2 diabetes mellitus with ketoacidosis\n│   ├── E1110: Type 2 diabetes mellitus with ketoacidosis without coma\n│   └── E1111: Type 2 diabetes mellitus with ketoacidosis with coma\n├── E112: Type 2 diabetes mellitus with kidney complications\n│   ├── E1121: Type 2 diabetes mellitus with diabetic nephropathy\n│   ├── E1122: Type 2 diabetes mellitus with diabetic chronic kidney disease\n│   └── E1129: Type 2 diabetes mellitus with other diabetic kidney complication\n├── E113: Type 2 diabetes mellitus with ophthalmic complications\n│   ├── E1131: Type 2 diabetes mellitus with unspecified diabetic retinopathy\n│   │   ├── E11311: Type 2 diabetes mellitus with unspecified diabetic retinopathy with ma

In [108]:
with open(os.getcwd().split("clincodex")[0]+"/system_prompts_v2.yaml", 'r') as f:
            system_prompts = yaml.safe_load(f)
            
system_prompt=system_prompts["DISEASE_CODING"]

In [109]:
code_agent_sliced = {k:str(v[2]) for k,v in coder_agent_pack.items()}
code_agent_sliced

{'E11': 'digraph {\n\tnode [shape=box style=rounded]\n\tE11 [label="E11: Type 2 diabetes mellitus"]\n\tE110 [label="E110: Type 2 diabetes mellitus with hyperosmolarity"]\n\tE11 -> E110\n\tE1100 [label="E1100: Type 2 diabetes mellitus with hyperosmolarity without nonketotic hyperglycemic-hyperosmolar coma (NKHHC) (billable)" fillcolor=lightblue style=filled]\n\tE110 -> E1100\n\tE1101 [label="E1101: Type 2 diabetes mellitus with hyperosmolarity with coma (billable)" fillcolor=lightblue style=filled]\n\tE110 -> E1101\n\tE111 [label="E111: Type 2 diabetes mellitus with ketoacidosis"]\n\tE11 -> E111\n\tE1110 [label="E1110: Type 2 diabetes mellitus with ketoacidosis without coma (billable)" fillcolor=lightblue style=filled]\n\tE111 -> E1110\n\tE1111 [label="E1111: Type 2 diabetes mellitus with ketoacidosis with coma (billable)" fillcolor=lightblue style=filled]\n\tE111 -> E1111\n\tE112 [label="E112: Type 2 diabetes mellitus with kidney complications"]\n\tE11 -> E112\n\tE1121 [label="E1121: T

In [110]:
import json

coder_agent_res=json.loads(client.claude_wrapper(system_prompt,user_input=f'''
                <<Text List >>
                {user_input}
        
                << Disease Dictionary >>
                {code_agent_sliced}
                
                '''))

{'input_tokens': 9105, 'output_tokens': 28}


In [111]:
coder_agent_res

['E1140', 'I10', 'J4530', 'N189', 'E1137X3']

In [112]:
final_res = {}
for code in coder_agent_res:
    final_res[code]=base_data_dict[code]


In [113]:
final_res

{'E1140': 'Type 2 diabetes mellitus with diabetic neuropathy, unspecified',
 'I10': 'Essential (primary) hypertension',
 'J4530': 'Mild persistent asthma, uncomplicated',
 'N189': 'Chronic kidney disease, unspecified',
 'E1137X3': 'Type 2 diabetes mellitus with diabetic macular edema, resolved following treatment, bilateral'}